# Traitement automatisé — toutes les visites (V0, V1, V3, V5, Vc)



In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from functools import reduce
import re

## classification des  psychotropes 

In [13]:
# ── Chargement des 2 fichiers CSV ──────────────────────────────────────
df1 = pd.read_csv("Output/psychotropes/psychotropes.csv", sep=";", encoding="utf-8")
df2 = pd.read_csv("Output/psychotropes/medicaments_classes_manuellement.csv", sep=";", encoding="utf-8")

# ── Concaténation ─────────────────────────────────────────────────────
dict_complet = pd.concat([df1, df2], ignore_index=True)

# ── Nettoyage ─────────────────────────────────────────────────────────
dict_complet.columns = dict_complet.columns.str.strip().str.upper()
dict_complet["NOM"] = dict_complet["NOM"].str.strip().str.upper()
dict_complet = dict_complet.dropna(subset=["NOM"]).drop_duplicates(subset=["NOM"])



In [14]:
def classer_medicaments(df, dict_complet):

    # ── Mapping NOM → CODE et NOM → CLASSEMENT ────────────────────────
    med_to_code       = dict(zip(dict_complet["NOM"], dict_complet["CODE"]))
    med_to_classement = dict(zip(dict_complet["NOM"], dict_complet["CLASSEMENT"]))

    # ── Liste des catégories uniques (pour les colonnes de comptage) ───
    categories = dict_complet["CLASSEMENT"].dropna().unique().tolist()

    def nettoyer_medicament(valeur):
        if pd.isna(valeur):
            return np.nan
        val = str(valeur).strip()
        if val in ("", "NAN", "<NA>"):
            return np.nan
        val = re.sub(r"^[\s\-\.\*_]+", "", val).strip()
        if not val:
            return np.nan
        if re.match(r"^\d+\s*$", val):
            return np.nan
        if re.match(r"^\.[A-Za-z0-9]{0,3}$", val):
            return np.nan
        if re.match(r"^[\s\-\.\*_/,;:]+$", val):
            return np.nan
        if re.match(r"^medicament\s*\d*$", val, re.IGNORECASE):
            return np.nan
        return val.strip().upper() or np.nan

    def get_classement(med_name):
        """Retourne le CLASSEMENT (catégorie textuelle) au lieu du CODE numérique."""
        if pd.isna(med_name):
            return np.nan
        med = str(med_name).strip().upper()
        if med in med_to_classement:
            return med_to_classement[med]
        premier_mot = re.split(r"[\s\(\)/\-,]", med)[0].strip()
        if premier_mot and premier_mot in med_to_classement:
            return med_to_classement[premier_mot]
        return np.nan

    med_cols = sorted(
        [c for c in df.columns if re.match(r"^MEDICMT\d+$", c, re.IGNORECASE)],
        key=lambda c: int(re.search(r"\d+$", c).group())
    )

    # ── Nettoyage + CLASS = CLASSEMENT catégoriel ─────────────────────
    for col in med_cols:
        n         = re.search(r"\d+$", col).group()
        poso_col  = f"POSO{n}"
        class_col = f"CLASS{n}"

        df[col]       = df[col].apply(nettoyer_medicament)
        df[class_col] = df[col].apply(get_classement)   # ← catégorie textuelle

        if poso_col in df.columns:
            poso_idx = df.columns.get_loc(poso_col)
            cols = df.columns.tolist()
            cols.remove(class_col)
            cols.insert(poso_idx + 1, class_col)
            df = df[cols]

    # ── Colonnes CLASS existantes ──────────────────────────────────────
    class_cols = []
    for col in med_cols:
        n = re.search(r"\d+$", col).group()
        c = f"CLASS{n}"
        if c in df.columns:
            class_cols.append(c)

    # ── Comptage par catégorie (une colonne par catégorie) ────────────
    for cat in categories:
        df[cat] = df[class_cols].apply(
            lambda row: (row == cat).sum(), axis=1
        )

    # ── nb_medicament = nombre de MEDICMT renseignés par patient ──────
    df["nb_medicament"] = df[med_cols].notna().sum(axis=1)

    # ── Rapport console ───────────────────────────────────────────────
    med_renseignes = df[med_cols].notna().sum().sum()
    non_cls_reels  = sum(
        (df[mc].notna() & df[cc].isna()).sum()
        for mc, cc in zip(med_cols, class_cols)
    )
    if med_renseignes > 0:
        pct = non_cls_reels / med_renseignes * 100
        print(f"  → Non classés : {non_cls_reels}/{med_renseignes} ({pct:.1f}%)")

    return df

---
##  nettoyage des codes manquants (".D", ".A", ".K", ".F", "D...")

In [6]:
def nettoyer_codes_manquants(df, prefixes=("." ,)):
    """
    Repère toutes les valeurs texte commençant par l'un des préfixes
    dans les colonnes object du DataFrame, puis les remplace par NaN.
    Retourne le DataFrame nettoyé (inplace).
    """
    text_cols = df.select_dtypes(include="object").columns
    all_unique = set()
    for col in text_cols:
        all_unique.update(df[col].dropna().unique())

    codes = [v for v in all_unique
             if any(str(v).startswith(p) for p in prefixes)]
    if codes:
        print(f"  → codes remplacés par NaN : {codes}")
    return df.replace(codes, np.nan)



## tableau_synthetique

In [15]:
def construire_tableau_synthetique(result, v):
    suffix = f"_V{v}"
    dfs = []

    # 1. Feuille Vx
    df_v = result[f"V{v}"].copy()
    df_v = df_v.drop(columns=[c for c in df_v.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
    df_v = df_v.rename(columns={c: f"{c}{suffix}" for c in df_v.columns if c != "SUBJID"})
    dfs.append(df_v)

    # 2. PDQ39_SI uniquement
    if "PDQ39" in result:
        df = result["PDQ39"][["SUBJID", "PDQ39_SI"]].copy()
        df = df.rename(columns={"PDQ39_SI": f"PDQ39_SI{suffix}"})
        dfs.append(df)

    # 3. UPDRSIII_S pour V0, sinon UPDRSIII
    key = "UPDRSIII_S" if (v == 0 and "UPDRSIII_S" in result) else "UPDRSIII"
    if key in result:
        df = result[key].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        if key =="UPDRSIII" and v != 1 :
            df=df.drop(columns=["UPDRSIII_tot","statut"])
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 4. UPDRSIV
    if "UPDRSIV" in result:
        df = result["UPDRSIV"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 5. ECMP
    if "ECMP" in result:
        df = result["ECMP"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 6. QUIP
    if "QUIP" in result:
        df = result["QUIP"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 7. LARS
    if "LARS" in result:
        df = result["LARS"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT","LARS_SCORE","statut","LARS_RESULTAT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 8. UPPS
    if "UPPS" in result:
        df = result["UPPS"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 9. DIGITSMT_TRAILMT_DKEFS
    if "DIGITSMT_TRAILMT_DKEFS" in result:
        df = result["DIGITSMT_TRAILMT_DKEFS"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # 10. HAMD_tot uniquement
    if "HAMD" in result:
        df = result["HAMD"][["SUBJID", "HAMD_tot"]].copy()
        df = df.rename(columns={"HAMD_tot": f"HAMD_tot{suffix}"})
        dfs.append(df)

    # 11. HAMA_tot uniquement
    if "HAMA" in result:
        df = result["HAMA"][["SUBJID", "HAMA_tot"]].copy()
        df = df.rename(columns={"HAMA_tot": f"HAMA_tot{suffix}"})
        dfs.append(df)

    # 12. MOCA_tot uniquement
    if "MOCA" in result:
        df = result["MOCA"][["SUBJID", "MOCA_tot"]].copy()
        df = df.rename(columns={"MOCA_tot": f"MOCA_tot{suffix}"})
        dfs.append(df)

    # 13. PSYCHOTROPES → colonnes CLASS* uniquement
    if "PSYCHOTROPES" in result:
        df = result["PSYCHOTROPES"][["SUBJID",'Anxiolytiques', 'Antidépresseurs','Neuroleptiques', 'Thymorégulateurs']].copy()
        df = df.rename(columns={
        'Anxiolytiques': f"Anxiolytiques{suffix}",
        'Antidépresseurs': f"Antidépresseurs{suffix}",
        'Neuroleptiques': f"Neuroleptiques{suffix}",
        'Thymorégulateurs': f"Thymorégulateurs{suffix}"
        })
        dfs.append(df)
        

    # 14. AUTRE_PARKINSON → colonnes CLASS* uniquement
    # if "AUTRE_PARKINSON" in result:
    #     df = result["AUTRE_PARKINSON"].copy()
    #     cols = ["SUBJID"] + [c for c in df.columns if c.upper().startswith("CLASS")]
    #     df = df[cols]
    #     df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
    #     dfs.append(df)

    # 15. LEDD
    if "LEDD" in result:
        df = result["LEDD"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE", "VISIT")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)
    
    if "FREQUENCE" in result :
        df = result["FREQUENCE"].copy()
        df = df.drop(columns=[c for c in df.columns if c.upper() in ("VISITE")], errors="ignore")
        df = df.rename(columns={c: f"{c}{suffix}" for c in df.columns if c != "SUBJID"})
        dfs.append(df)

    # ── Fusion sur SUBJID ─────────────────────────────────────────────
    df_final = dfs[0]
    for df_next in dfs[1:]:
        df_final = pd.merge(df_final, df_next, on="SUBJID", how="outer")

    return df_final

---
## Fonction principale : `traiter_visite(v, vc=False)`

In [16]:
def traiter_visite(v, vc=False):

    chemin = f"Output/version_2/V{v}.xlsx"
    print(f"\n{'='*60}")
    print(f"  TRAITEMENT  V{v}  —  {chemin}")
    print(f"{'='*60}")

    # ==================================================================
    # FEUILLES COMMUNES  (présentes dans toutes les visites, y compris Vc)
    # ==================================================================
     
    df_V = pd.read_excel(chemin, sheet_name=f"V{v}")
    df_V = nettoyer_codes_manquants(df_V)

    # ── LEDD ──────────────────────────────────────────────────────────
    df_LEDD = pd.read_excel(chemin, sheet_name="LEDD")
    df_LEDD["somme_calc"] = df_LEDD.iloc[:, 2:-1].sum(axis=1, skipna=True)
    df_LEDD["statut"] = np.where(
        np.isclose(df_LEDD["somme_calc"], df_LEDD["ledd_tot"], atol=0.01),
        "ok", "différent"
    )
    df_LEDD["ledd_tot"] = df_LEDD["somme_calc"]
    df_LEDD.drop(columns=["somme_calc", "statut"], inplace=True)
    df_Total_LEDD = df_LEDD[["SUBJID", "ledd_tot"]]
    print(f"df_LEDD                 : {df_LEDD.shape}")

    # ── PSYCHOTROPES ──────────────────────────────────────────────────
    df_PSYCHOTROPES = pd.read_excel(chemin, sheet_name="PSYCHOTROPES")
    print(f"\nFeuille PSYCHOTROPES    : {df_PSYCHOTROPES.shape}")
    df_PSYCHOTROPES = nettoyer_codes_manquants(df_PSYCHOTROPES)
    cols = df_PSYCHOTROPES.columns[:2].tolist()
    cols += [c for c in df_PSYCHOTROPES.columns if c.startswith('MEDICMT') or c.startswith('POSO')]
    df_PSYCHOTROPES = df_PSYCHOTROPES[cols]
    df_PSYCHOTROPES = classer_medicaments(df_PSYCHOTROPES, dict_complet)
    print(f"df_PSYCHOTROPES         : {df_PSYCHOTROPES.shape}")

    # ── AUTRE_PARKINSON ───────────────────────────────────────────────
    df_AUTRE_PARKINSON = pd.read_excel(chemin, sheet_name="AUTRE_PARKINSON")
    df_AUTRE_PARKINSON = nettoyer_codes_manquants(df_AUTRE_PARKINSON)
    cols = df_AUTRE_PARKINSON.columns[:2].tolist()
    cols += [c for c in df_AUTRE_PARKINSON.columns if c.startswith('MEDICMT') or c.startswith('POSO')]
    df_AUTRE_PARKINSON = df_AUTRE_PARKINSON[cols]
    df_AUTRE_PARKINSON = classer_medicaments(df_AUTRE_PARKINSON, dict_complet)
    print(f"df_AUTRE_PARKINSON      : {df_AUTRE_PARKINSON.shape}")

    # ── CONSO_SPECIFIQUE ──────────────────────────────────────────────
    df_CONSO_SPECIFIQUE = pd.read_excel(chemin, sheet_name="CONSO_SPECIFIQUE")
    df_CONSO_SPECIFIQUE = nettoyer_codes_manquants(df_CONSO_SPECIFIQUE)
    cols = df_CONSO_SPECIFIQUE.columns[:2].tolist()
    cols += [c for c in df_CONSO_SPECIFIQUE.columns if c.startswith('MEDICMT') or c.startswith('POSO')]
    df_CONSO_SPECIFIQUE = df_CONSO_SPECIFIQUE[cols]
    print(f"df_CONSO_SPECIFIQUE     : {df_CONSO_SPECIFIQUE.shape}")

    # ==================================================================
    # CAS SPÉCIAL  →  Vc  (uniquement les feuilles communes)
    # ==================================================================
    # if vc:
    #     return {
    #         f"V{v}"             : df_V,
    #         "LEDD"             : df_LEDD,
    #         "CONSO_SPECIFIQUE" : df_CONSO_SPECIFIQUE,
    #         "PSYCHOTROPES"     : df_PSYCHOTROPES,
    #         "AUTRE_PARKINSON"  : df_AUTRE_PARKINSON,
    #     }
        
    if vc:
        result = {
            f"V{v}"             : df_V,
            "LEDD"             : df_LEDD,
            "CONSO_SPECIFIQUE" : df_CONSO_SPECIFIQUE,
            "PSYCHOTROPES"     : df_PSYCHOTROPES,
            "AUTRE_PARKINSON"  : df_AUTRE_PARKINSON,
        }

        result["SYNTHESE"] = construire_tableau_synthetique(result, v)

        return result
    # ==================================================================
    # FEUILLES SPÉCIFIQUES  V0 → V5
    # ==================================================================

    # ── UPDRSIII ──────────────────────────────────────────────────────
    if v in (0, 1):
        df_UPDRSIII = pd.read_excel(chemin, sheet_name="UPDRSIII")
        print(f"\nFeuille UPDRSIII        : {df_UPDRSIII.shape}")

        df_UPDRSIII = nettoyer_codes_manquants(df_UPDRSIII, prefixes=(".",))
        df_UPDRSIII = nettoyer_codes_manquants(df_UPDRSIII, prefixes=("D",))

        n_last = 7 if v == 0 else 4
        df_Total_UPDRSIII = df_UPDRSIII.iloc[:, [1] + list(range(-n_last, 0))]
        print(f"df_UPDRSIII             : {df_UPDRSIII.shape}")

        # ── UPDRSIII_S  (uniquement V0) ───────────────────────────────
        # Calcul du BEST ON, dopasensibilité, à partir de df_UPDRSIII
        if v == 0:
            TEMPS = [15, 30, 45, 60, 90, 120]

            # Mapping Convention B : suffixe index → temps en minutes
            SUFFIXE_B_TO_TEMPS = {"": 15, "1": 30, "2": 45, "3": 60, "4": 90, "5": 120}
            TEMPS_TO_SUFFIXE_B = {v_: k for k, v_ in SUFFIXE_B_TO_TEMPS.items()}

            df_UPDRSIII_S = df_UPDRSIII.copy()
            df_cols = list(df_UPDRSIII_S.columns)

            # Préfixes à exclure de la détection d'items
            EXCLUDED_PREFIXES = ("SS", "TOT", "SSTOT")

            # Détecter les items à partir des colonnes OFF_* (hors OFF_H et SS*/TOT*/SSTOT*)
            import re
            items = [
                re.sub(r'^OFF_', '', c, flags=re.IGNORECASE)
                for c in df_cols
                if re.match(r'^OFF_', c, re.IGNORECASE)
                and c.upper() != 'OFF_H'
                and not any(c.upper().startswith(p) for p in EXCLUDED_PREFIXES)
            ]

            def get_on_col(item, t):
                """
                Retourne le nom de colonne ON pour un item et un temps donnés.
                Teste d'abord la Convention A (ON_{item}15/30/.../120),
                puis la Convention B (ON_{item} / ON_{item}1 / ... / ON_{item}5).
                Retourne None si aucune colonne trouvée.
                """
                # Convention A : ex. ON_MIGCHE_PIED30
                col_a = f"ON_{item}{t}"
                if col_a in df_cols:
                    return col_a
                # Convention B : ex. ON_MIDROIT_JAMBE1 pour t=30
                suffix_b = TEMPS_TO_SUFFIXE_B.get(t, None)
                if suffix_b is not None:
                    col_b = f"ON_{item}{suffix_b}"
                    if col_b in df_cols:
                        return col_b
                return None

            # Convertir toutes les colonnes de scores en numérique
            off_cols = [f"OFF_{item}" for item in items if f"OFF_{item}" in df_cols]
            on_cols_all = [
                get_on_col(item, t)
                for item in items for t in TEMPS
                if get_on_col(item, t) is not None
            ]
            score_cols = off_cols + on_cols_all
            df_UPDRSIII_S[score_cols] = df_UPDRSIII_S[score_cols].apply(pd.to_numeric, errors="coerce")

            # 1. Score total OFF
            df_UPDRSIII_S["SCORE_TOTAL_OFF"] = df_UPDRSIII_S[off_cols].sum(axis=1, min_count=1)

            # 2. Scores totaux ON par temps
            for t in TEMPS:
                on_cols_t = [get_on_col(item, t) for item in items if get_on_col(item, t) is not None]
                if on_cols_t:
                    df_UPDRSIII_S[f"SCORE_TOTAL_ON_{t}"] = df_UPDRSIII_S[on_cols_t].sum(axis=1, min_count=1)

            # 3 & 4. BEST ON et temps correspondant
            on_total_cols = [f"SCORE_TOTAL_ON_{t}" for t in TEMPS if f"SCORE_TOTAL_ON_{t}" in df_UPDRSIII_S.columns]
            on_matrix = df_UPDRSIII_S[on_total_cols]
            df_UPDRSIII_S["BEST_ON"]      = on_matrix.min(axis=1)
            best_col                       = on_matrix.idxmin(axis=1)   # ex: "SCORE_TOTAL_ON_30"
            df_UPDRSIII_S["BEST_ON_TEMPS"] = best_col.str.extract(r'(\d+)$').astype(float)

            # 5. Conserver les items ON au temps BEST ON → colonnes BEST_ON_{item}
            all_on_item_cols = []
            for item in items:
                best_vals = []
                for _, row in df_UPDRSIII_S.iterrows():
                    best_t = row.get("BEST_ON_TEMPS")
                    if pd.isna(best_t):
                        best_vals.append(np.nan)
                        continue
                    col_name = get_on_col(item, int(best_t))
                    best_vals.append(row[col_name] if col_name and col_name in row.index else np.nan)
                df_UPDRSIII_S[f"BEST_ON_{item}"] = best_vals
                # Marquer toutes les colonnes ON_{item}{t} pour suppression
                for t in TEMPS:
                    c = get_on_col(item, t)
                    if c and c in df_UPDRSIII_S.columns:
                        all_on_item_cols.append(c)

            # Supprimer les colonnes ON par temps (maintenant redondantes)
            df_UPDRSIII_S.drop(columns=list(set(all_on_item_cols)), errors="ignore", inplace=True)

            # 6. Dopasensibilité
            off_num = pd.to_numeric(df_UPDRSIII_S["SCORE_TOTAL_OFF"], errors="coerce")
            bon_num = pd.to_numeric(df_UPDRSIII_S["BEST_ON"],         errors="coerce")
            df_UPDRSIII_S["DOPASENSIBILITE_PCT"] = np.where(
                off_num == 0,
                np.nan,
                (off_num - bon_num) / off_num * 100
            )

            # Supprimer les colonnes SS*, TOT*, SSTOT*
            cols_to_drop = [c for c in df_UPDRSIII_S.columns
                            if re.match(r'^(SS|TOT|SSTOT)', c, re.IGNORECASE)]
            df_UPDRSIII_S.drop(columns=cols_to_drop, errors="ignore", inplace=True)

            print(f"df_UPDRSIII_S           : {df_UPDRSIII_S.shape}")

    elif v in (3, 5):
        df_raw_v3v5 = pd.read_excel(
            "Data/Matthieu_Soumaya_Dec2025.xlsx",
            sheet_name="UPDRSIII_COMPLET_V3_V5 "
        )
        label_map = {
            3: "Visite Bilan à 3 ans - V3",
            5: "Visite Bilan à 5 ans - V5",
        }
        df_UPDRSIII = df_raw_v3v5[
            df_raw_v3v5["VISIT"] == label_map[v]
        ].reset_index(drop=True)

        df_UPDRSIII = nettoyer_codes_manquants(df_UPDRSIII, prefixes=(".",))
        df_UPDRSIII = nettoyer_codes_manquants(df_UPDRSIII, prefixes=("D",))

        # Supprimer les 2 dernières colonnes redondantes
        # df_UPDRSIII.drop(columns=df_UPDRSIII.iloc[:, -2:].columns, inplace=True)
        df_UPDRSIII.drop("VISIT",axis=1,inplace=True)
        df_UPDRSIII.insert(0,"VISITE",v)

        cols_num = df_UPDRSIII.iloc[:, 4:].columns
        df_UPDRSIII[cols_num] = df_UPDRSIII[cols_num].apply(pd.to_numeric, errors="coerce")

        cols_score = df_UPDRSIII.iloc[:, 4:-2].columns
        cols_score = cols_score.drop(["SS_TOTAL1","ON_SS_TOT1"], errors="ignore")
        df_UPDRSIII["UPDRSIII_tot"] = df_UPDRSIII[cols_score].sum(axis=1, skipna=True)
        df_UPDRSIII["statut"] = np.where(
            np.isclose(df_UPDRSIII["UPDRSIII_tot"], df_UPDRSIII["ON_TOTAL"], atol=0.01),
            "ok", "différent"
        )
        df_Total_UPDRSIII = df_UPDRSIII[["SUBJID","UPDRSIII_tot"]]
        print(f"\ndf_UPDRSIII (V{v})       : {df_UPDRSIII.shape}")

    # ── IMC ───────────────────────────────────────────────────────────
    cols_v = df_V.iloc[:, -2:].columns
    df_V[cols_v] = df_V[cols_v].apply(pd.to_numeric, errors="coerce")
    df_V["TAILLE_m"] = np.where(df_V["TAILLE"] > 3, df_V["TAILLE"] / 100, df_V["TAILLE"])
    df_V["IMC"] = df_V["POIDS"] / (df_V["TAILLE_m"] ** 2)
    df_V.drop("TAILLE_m", axis=1, inplace=True)



    # ── UPDRSIV ───────────────────────────────────────────────────────
    df_UPDRSIV = pd.read_excel(chemin, sheet_name="UPDRSIV")
    print(f"\nFeuille UPDRSIV         : {df_UPDRSIV.shape}")
    df_UPDRSIV = nettoyer_codes_manquants(df_UPDRSIV)
    mapping_UPDRSIV = {"Normal": 0, "Minime": 1, "Léger": 2, "Modéré": 3, "Sévère": 4}
    df_UPDRSIV.iloc[:, 3:] = df_UPDRSIV.iloc[:, 3:].replace(mapping_UPDRSIV)
    df_UPDRSIV["UPDRSIV_tot"] = df_UPDRSIV.iloc[:, 2:].sum(axis=1, skipna=True)
    df_Total_UPDRSIV = df_UPDRSIV[["SUBJID", "UPDRSIV_tot"]]
    print(f"df_UPDRSIV              : {df_UPDRSIV.shape}")

    # ── PDQ39 ─────────────────────────────────────────────────────────
    df_PDQ39 = pd.read_excel(chemin, sheet_name="PDQ39")
    print(f"\nFeuille PDQ39           : {df_PDQ39.shape}")
    df_PDQ39 = nettoyer_codes_manquants(df_PDQ39)
    mapping_PDQ39 = {
        "Jamais": 0, "Rarement": 1, "Parfois": 2, "Souvent": 3,
        "Toujours": 4, "Toujours ou ne peut jamais faire": 4,
        "Oui": 1, "Non": 0
    }
    cols_obj = df_PDQ39.select_dtypes(include="object").columns[1:]
    df_PDQ39[cols_obj] = df_PDQ39[cols_obj].replace(mapping_PDQ39)
    domains = {
        "mobility":      ["PDQ39_1","PDQ39_2","PDQ39_3","PDQ39_4","PDQ39_5","PDQ39_6","PDQ39_7","PDQ39_8","PDQ39_9","PDQ39_10"],
        "adl":           ["PDQ39_11","PDQ39_12","PDQ39_13","PDQ39_14","PDQ39_15","PDQ39_16"],
        "emotional":     ["PDQ39_17","PDQ39_18","PDQ39_19","PDQ39_20","PDQ39_21","PDQ39_22"],
        "stigma":        ["PDQ39_23","PDQ39_24","PDQ39_25","PDQ39_26"],
        "social":        ["PDQ39_27","PDQ39_28","PDQ39_29"],
        "cognition":     ["PDQ39_30","PDQ39_31","PDQ39_32","PDQ39_33"],
        "communication": ["PDQ39_34","PDQ39_35","PDQ39_36"],
        "bodily":        ["PDQ39_37","PDQ39_38","PDQ39_39"]
    }
    for domain, cols in domains.items():
        df_PDQ39[domain + "_score"] = df_PDQ39[cols].sum(axis=1) / (len(cols) * 4) * 100
    domain_cols = [d + "_score" for d in domains.keys()]
    df_PDQ39["PDQ39_SI"] = df_PDQ39[domain_cols].mean(axis=1)
    df_Total_PDQ39 = df_PDQ39[["SUBJID", "PDQ39_SI"]]
    print(f"df_PDQ39                : {df_PDQ39.shape}")

    # ── QUIP ──────────────────────────────────────────────────────────
    df_QUIP = pd.read_excel(chemin, sheet_name="QUIP")
    print(f"\nFeuille QUIP            : {df_QUIP.shape}")
    df_QUIP = nettoyer_codes_manquants(df_QUIP)

    # ── MOCA ──────────────────────────────────────────────────────────
    df_MOCA = pd.read_excel(chemin, sheet_name="MOCA")
    print(f"\nFeuille MOCA            : {df_MOCA.shape}")
    df_MOCA = nettoyer_codes_manquants(df_MOCA)
    cols_moca = df_MOCA.iloc[:, 2:].columns
    df_MOCA[cols_moca] = df_MOCA[cols_moca].apply(pd.to_numeric, errors="coerce")
    df_MOCA["MOCA_tot"] = df_MOCA.iloc[:, 2:-1].sum(axis=1, skipna=True)
    df_MOCA["statut"] = np.where(
        np.isclose(df_MOCA["MOCA_tot"], df_MOCA["MOCA_SCORE"], atol=0.01),
        "ok", "différent"
    )
    df_Total_MOCA = df_MOCA[["SUBJID","MOCA_tot"]]
    print(f"df_MOCA                 : {df_MOCA.shape}")

    # ── HAMA ──────────────────────────────────────────────────────────
    df_HAMA = pd.read_excel(chemin, sheet_name="HAMA")
    print(f"\nFeuille HAMA            : {df_HAMA.shape}")
    df_HAMA = nettoyer_codes_manquants(df_HAMA)
    mapping_HAMA = {
        "Absent": 0,
        "Léger": 1, "Anxiété légère": 1,
        "Modéré": 2, "Anxiété légère à modérée": 2,
        "Sévère": 3, "Anxiété modérée à grave": 3,
        "Très sévère": 4
    }
    cols_obj = df_HAMA.select_dtypes(include="object").columns[1:]
    df_HAMA[cols_obj] = df_HAMA[cols_obj].replace(mapping_HAMA)
    cols_hama = df_HAMA.iloc[:, 2:-2].columns
    df_HAMA[cols_hama] = df_HAMA[cols_hama].apply(pd.to_numeric, errors="coerce")
    df_HAMA["HAMA_SCORECALC"] = df_HAMA["HAMA_SCORECALC"].apply(pd.to_numeric, errors="coerce")
    df_HAMA["HAMA_tot"] = df_HAMA[cols_hama].sum(axis=1, skipna=True)
    df_HAMA["statut"] = np.where(
        np.isclose(df_HAMA["HAMA_tot"], df_HAMA["HAMA_SCORECALC"], atol=0.01),
        "ok", "différent"
    )
    col = df_HAMA.pop("ANXIETE")
    df_HAMA["ANXIETE"] = col
    df_Total_HAMA = df_HAMA[["SUBJID","HAMA_tot"]]
    print(f"df_HAMA                 : {df_HAMA.shape}")

    # ── HAMD ──────────────────────────────────────────────────────────
    df_HAMD = pd.read_excel(chemin, sheet_name="HAMD")
    print(f"\nFeuille HAMD            : {df_HAMD.shape}")
    df_HAMD = nettoyer_codes_manquants(df_HAMD)
    cols_hamd = df_HAMD.iloc[:, 2:-2].columns
    df_HAMD["HAMD_tot"] = df_HAMD[cols_hamd].sum(axis=1, skipna=True)
    df_HAMD["statut"] = np.where(
        np.isclose(df_HAMD["HAMD_tot"], df_HAMD["HAMD_SCORECALC"], atol=0.01),
        "ok", "différent"
    )
    col = df_HAMD.pop("DEPRESSION")
    df_HAMD["DEPRESSION"] = col
    mapping_HAMD = {
        "Symptomes dépressifs légers": 1,
        "Symptômes dépressifs légers à modérés": 2,
        "Symptômes dépressifs modérés à sévères": 3,
    }
    cols_obj = df_HAMD.select_dtypes(include="object").columns[-1:]
    df_HAMD[cols_obj] = df_HAMD[cols_obj].replace(mapping_HAMD)
    df_Total_HAMD = df_HAMD[["SUBJID", "HAMD_tot"]]
    print(f"df_HAMD                 : {df_HAMD.shape}")

    # ── LARS ──────────────────────────────────────────────────────────
    df_LARS = pd.read_excel(chemin, sheet_name="LARS")
    print(f"\nFeuille LARS            : {df_LARS.shape}")
    df_LARS = nettoyer_codes_manquants(df_LARS)
    cols_lars = df_LARS.iloc[:, 2:-2].columns
    df_LARS["LARS_tot"] = df_LARS[cols_lars].sum(axis=1, skipna=True)
    df_LARS["statut"] = np.where(
        np.isclose(df_LARS["LARS_tot"], df_LARS["LARS_SCORE"], atol=0.01),
        "ok", "différent"
    )
    col = df_LARS.pop("LARS_RESULTAT")
    df_LARS["LARS_RESULTAT"] = col
    mapping_LARS = {
        "Non apathique": 1,
        "Tendance à l'apathie": 2,
        "Apathie modérée": 3,
        "Apathie sévère": 4,
    }
    cols_obj = df_LARS.select_dtypes(include="object").columns[-1:]
    df_LARS[cols_obj] = df_LARS[cols_obj].replace(mapping_LARS)
    df_Total_LARS = df_LARS[["SUBJID", "LARS_tot"]]
    print(f"df_LARS                 : {df_LARS.shape}")

    # ── ECMP ──────────────────────────────────────────────────────────
    df_ECMP = pd.read_excel(chemin, sheet_name="ECMP")
    print(f"\nFeuille ECMP            : {df_ECMP.shape}")
    df_ECMP = nettoyer_codes_manquants(df_ECMP)

    # ── DIGITSMT_TRAILMT_DKEFS ────────────────────────────────────────
    df_DIGITSMT = pd.read_excel(chemin, sheet_name="DIGITSMT_TRAILMT_DKEFS")
    print(f"\nFeuille DIGITSMT        : {df_DIGITSMT.shape}")
    df_DIGITSMT = nettoyer_codes_manquants(df_DIGITSMT)

    # ── UPPS (fichier externe, V0 V1 V3 V5) ──────────────────────────
    if v in (0, 1, 3, 5):
        label_map_upps = {
            0: "Visite de screening",
            1: "Visite Bilan à 1 an - V1",
            3: "Visite Bilan à 3 ans - V3",
            5: "Visite Bilan à 5 ans - V5",
        }
        df_raw_upps = pd.read_excel(
            "Data/Matthieu_Soumaya_Dec2025.xlsx",
            sheet_name="UPPS"
        )
        df_UPPS = df_raw_upps[
            df_raw_upps["TITRE"] == label_map_upps[v]
        ].reset_index(drop=True)
        df_UPPS.drop(columns=["TITRE", "INIT_PAT"], inplace=True)
        df_UPPS.insert(0, "VISITE", v)
        df_UPPS = nettoyer_codes_manquants(df_UPPS)
        print(f"df_UPPS                 : {df_UPPS.shape}")

    # ── FREQUENCE  (V1, V2, V3) ───────────────────────────────────────
    if v in (1, 2, 3):
        df_FREQUENCE = pd.read_excel(chemin, sheet_name="FREQUENCE")
        df_FREQUENCE = nettoyer_codes_manquants(df_FREQUENCE)

    # ==================================================================
    # TABLE DES TOTAUX
    # ==================================================================
    lis_totaux = [
        df_Total_MOCA,
        df_Total_PDQ39,
        df_Total_HAMA,
        df_Total_HAMD,    
    ]
    lis_avec_id = [d for d in lis_totaux if "SUBJID" in d.columns]
    if lis_avec_id:
        df_Totaux = reduce(
            lambda left, right: pd.merge(left, right, on="SUBJID", how="outer"),
            lis_avec_id
        )
        df_Totaux = pd.concat(
            [df_Totaux.iloc[[-1]], df_Totaux.iloc[:-1]], ignore_index=True
        )
    else:
        df_Totaux = pd.DataFrame()

    # ==================================================================
    # CONSTRUCTION DU DICTIONNAIRE DE RETOUR
    # ==================================================================
    result = {
        f"V{v}"                   : df_V,
        "LEDD"                    : df_LEDD,
        "CONSO_SPECIFIQUE"        : df_CONSO_SPECIFIQUE,
        "PSYCHOTROPES"            : df_PSYCHOTROPES,
        "AUTRE_PARKINSON"         : df_AUTRE_PARKINSON,
        "UPDRSIII"                : df_UPDRSIII,
        "UPDRSIV"                 : df_UPDRSIV,
        "PDQ39"                   : df_PDQ39,
        "QUIP"                    : df_QUIP,
        "MOCA"                    : df_MOCA,
        "HAMA"                    : df_HAMA,
        "HAMD"                    : df_HAMD,
        "LARS"                    : df_LARS,
        "ECMP"                    : df_ECMP,
        "DIGITSMT_TRAILMT_DKEFS"  : df_DIGITSMT,
        "Totaux"                  : df_Totaux,
    }

    # ── UPDRSIII_S  →  ajoutée juste après UPDRSIII, uniquement pour V0 ──
    if v == 0:
        result["UPDRSIII_S"] = df_UPDRSIII_S

    if v in (0, 1, 3, 5):
        result["UPPS"] = df_UPPS

    if v in (1, 2, 3):
        result["FREQUENCE"] = df_FREQUENCE
    
    # ── SYNTHESE ──────────────────────────────────────────────────────
    result["SYNTHESE"] = construire_tableau_synthetique(result, v)


    return result

In [17]:
ORDRE_FEUILLES = [
    
    "PDQ39",
    "UPDRSIII",
    "UPDRSIII_S",
    "UPDRSIV",
    
    "ECMP",
    "QUIP",
    "LARS",
    "UPPS",
    "HAMD",
    "HAMA",
    "MOCA",
    "DIGITSMT_TRAILMT_DKEFS",

    "FREQUENCE",
    
    "CONSO_SPECIFIQUE",
    "PSYCHOTROPES",
    "AUTRE_PARKINSON",
    
    "LEDD",
    "FREQUENCE"
    "Totaux",
    "SYNTHESE"
]

def reordonner_feuilles(sheets, v):
    cle_visite    = f"V{v}"
    ordre_complet = [cle_visite] + ORDRE_FEUILLES
    return {k: sheets[k] for k in ordre_complet if k in sheets}

---
## Utilitaire d'écriture Excel

In [18]:
def ecrire_excel(sheets_dict, version, output_dir="Output/version_3"):
    filepath = f"{output_dir}/V{version}.xlsx"
    with pd.ExcelWriter(filepath, engine="openpyxl") as writer:
        for sheet_name, df in sheets_dict.items():
            df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    print(f"[OK] {filepath}  →  {len(sheets_dict)} feuilles")

---
## Traitement + écriture de chaque visite

### Vc

In [19]:
sheets_Vc = traiter_visite("c", vc=True)
sheets_Vc = reordonner_feuilles(sheets_Vc, "c")
ecrire_excel(sheets_Vc, "c")


  TRAITEMENT  Vc  —  Output/version_2/Vc.xlsx
df_LEDD                 : (491, 7)

Feuille PSYCHOTROPES    : (835, 80)
  → codes remplacés par NaN : ['.A', '.D']


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


  → Non classés : 67/385 (17.4%)
df_PSYCHOTROPES         : (835, 46)
  → codes remplacés par NaN : ['.A', '.D', '.C', '.K']
  → Non classés : 220/264 (83.3%)
df_AUTRE_PARKINSON      : (835, 46)
  → codes remplacés par NaN : ['.A']
df_CONSO_SPECIFIQUE     : (835, 28)


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


[OK] Output/version_3/Vc.xlsx  →  6 feuilles


### V0

In [20]:
sheets_V0 = traiter_visite(0)
sheets_V0 = reordonner_feuilles(sheets_V0, 0)

ecrire_excel(sheets_V0, 0)



  TRAITEMENT  V0  —  Output/version_2/V0.xlsx
  → codes remplacés par NaN : ['.D']
df_LEDD                 : (794, 7)

Feuille PSYCHOTROPES    : (835, 80)
  → codes remplacés par NaN : ['.D', '.K', '.A']


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


  → Non classés : 49/524 (9.4%)
df_PSYCHOTROPES         : (835, 46)
  → codes remplacés par NaN : ['.D', '.C', '.K', '.A']
  → Non classés : 432/499 (86.6%)
df_AUTRE_PARKINSON      : (835, 46)
  → codes remplacés par NaN : ['.D', '.K', '.A']
df_CONSO_SPECIFIQUE     : (835, 28)


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)



Feuille UPDRSIII        : (836, 265)
  → codes remplacés par NaN : ['.D', '.F', '.K', '.A']
  → codes remplacés par NaN : ['DM:DM']
df_UPDRSIII             : (836, 265)


...


Feuille HAMD            : (835, 22)
df_HAMD                 : (835, 24)


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\99602970.py:345: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_HAMD[cols_obj] = df_HAMD[cols_obj].replace(mapping_HAMD)



Feuille LARS            : (835, 13)
df_LARS                 : (835, 15)


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\99602970.py:368: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_LARS[cols_obj] = df_LARS[cols_obj].replace(mapping_LARS)



Feuille ECMP            : (835, 44)
  → codes remplacés par NaN : ['.A', '.D']

Feuille DIGITSMT        : (835, 21)
  → codes remplacés par NaN : ['.A', '.D', '.F', '.C', '.K']
df_UPPS                 : (835, 22)
[OK] Output/version_3/V0.xlsx  →  18 feuilles


### V1

In [21]:
sheets_V1 = traiter_visite(1)
sheets_V1 = reordonner_feuilles(sheets_V1, 1)
ecrire_excel(sheets_V1, 1)


  TRAITEMENT  V1  —  Output/version_2/V1.xlsx
  → codes remplacés par NaN : ['.D', '.A']
df_LEDD                 : (525, 7)

Feuille PSYCHOTROPES    : (835, 80)
  → codes remplacés par NaN : ['.A', '.D']
  → Non classés : 46/387 (11.9%)
df_PSYCHOTROPES         : (835, 46)
  → codes remplacés par NaN : ['.A', '.K']


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


  → Non classés : 191/225 (84.9%)
df_AUTRE_PARKINSON      : (835, 46)
  → codes remplacés par NaN : ['.D', '.A']
df_CONSO_SPECIFIQUE     : (835, 28)


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)



Feuille UPDRSIII        : (836, 155)
  → codes remplacés par NaN : ['.D', '.F', '.K', '.A']
  → codes remplacés par NaN : ['Dose de L Dopa', 'DM:DM', 'DM']
df_UPDRSIII             : (836, 155)

Feuille UPDRSIV         : (835, 9)
  → codes remplacés par NaN : ['.D']
df_UPDRSIV              : (835, 10)


...


Feuille LARS            : (835, 13)
df_LARS                 : (835, 15)


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\99602970.py:368: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_LARS[cols_obj] = df_LARS[cols_obj].replace(mapping_LARS)



Feuille ECMP            : (835, 44)
  → codes remplacés par NaN : ['.A', '.D']

Feuille DIGITSMT        : (835, 21)
  → codes remplacés par NaN : ['.A', '.D', '.F', '.K']
df_UPPS                 : (835, 22)
  → codes remplacés par NaN : ['.D', '.F', '.C', '.K', '.6', '.A']


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


[OK] Output/version_3/V1.xlsx  →  18 feuilles


### V3

In [22]:
sheets_V3 = traiter_visite(3)
sheets_V3 = reordonner_feuilles(sheets_V3, 3)
ecrire_excel(sheets_V3, 3)


  TRAITEMENT  V3  —  Output/version_2/V3.xlsx
  → codes remplacés par NaN : ['.D', '.A']
df_LEDD                 : (272, 7)

Feuille PSYCHOTROPES    : (835, 80)
  → codes remplacés par NaN : ['.A', '.D']


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


  → Non classés : 20/169 (11.8%)
df_PSYCHOTROPES         : (835, 46)
  → codes remplacés par NaN : ['.A', '.D']


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


  → Non classés : 96/120 (80.0%)
df_AUTRE_PARKINSON      : (835, 46)
  → codes remplacés par NaN : ['.D', '.K', '.A']
df_CONSO_SPECIFIQUE     : (835, 28)


...


Feuille QUIP            : (835, 33)

Feuille MOCA            : (835, 14)
df_MOCA                 : (835, 16)

Feuille HAMA            : (835, 18)
df_HAMA                 : (835, 20)


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\99602970.py:313: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_HAMA[cols_obj] = df_HAMA[cols_obj].replace(mapping_HAMA)



Feuille HAMD            : (835, 22)
df_HAMD                 : (835, 24)

Feuille LARS            : (835, 13)
df_LARS                 : (835, 15)


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\99602970.py:345: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_HAMD[cols_obj] = df_HAMD[cols_obj].replace(mapping_HAMD)
C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\99602970.py:368: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_LARS[cols_obj] = df_LARS[cols_obj].replace(mapping_LARS)



Feuille ECMP            : (835, 44)
  → codes remplacés par NaN : ['.D']

Feuille DIGITSMT        : (835, 21)
  → codes remplacés par NaN : ['.A', '.D', '.F', '.K']
df_UPPS                 : (835, 22)
  → codes remplacés par NaN : ['.D', '.F', '.K', '.A']
[OK] Output/version_3/V3.xlsx  →  18 feuilles


### V5

In [23]:
sheets_V5 = traiter_visite(5)
sheets_V5 = reordonner_feuilles(sheets_V5, 5)
ecrire_excel(sheets_V5, 5)


  TRAITEMENT  V5  —  Output/version_2/V5.xlsx
  → codes remplacés par NaN : ['.D', '.A']
df_LEDD                 : (313, 7)

Feuille PSYCHOTROPES    : (835, 80)
  → codes remplacés par NaN : ['.A', '.D', '.K']


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


  → Non classés : 24/241 (10.0%)
df_PSYCHOTROPES         : (835, 46)
  → codes remplacés par NaN : ['.A', '.D', '.K']


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


  → Non classés : 166/193 (86.0%)
df_AUTRE_PARKINSON      : (835, 46)
  → codes remplacés par NaN : ['.D', '.K', '.A']
df_CONSO_SPECIFIQUE     : (835, 28)


...


Feuille QUIP            : (835, 33)

Feuille MOCA            : (835, 14)
df_MOCA                 : (835, 16)

Feuille HAMA            : (835, 18)
df_HAMA                 : (835, 20)


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\99602970.py:313: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_HAMA[cols_obj] = df_HAMA[cols_obj].replace(mapping_HAMA)



Feuille HAMD            : (835, 22)
df_HAMD                 : (835, 24)

Feuille LARS            : (835, 13)


C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\99602970.py:345: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_HAMD[cols_obj] = df_HAMD[cols_obj].replace(mapping_HAMD)
C:\Users\toufi\AppData\Local\Temp\ipykernel_19532\99602970.py:368: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_LARS[cols_obj] = df_LARS[cols_obj].replace(mapping_LARS)


df_LARS                 : (835, 15)

Feuille ECMP            : (835, 44)
  → codes remplacés par NaN : ['.D']

Feuille DIGITSMT        : (835, 21)
  → codes remplacés par NaN : ['.A', '.D', '.F', '.K']
df_UPPS                 : (835, 22)
[OK] Output/version_3/V5.xlsx  →  17 feuilles


#

# Prétraitement info statiques 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
info =  pd.read_excel("Output/version_2/info.xlsx")
info.head()


,SUBJID,INIT_PAT,D_SCREEN,D_1ER_SYMPT,D_DIAG,D_LDOPA,D_TTT_DOPAM,D_FLUCTU_MOTR,D_FLUCTU_NONMOTR,D_DYSKINESIE,...,AGE,SEXE,SUIVI_ETUDE,D_FIN_ETUDE,D_SORTIE_PREMA,MOTIF_SORTIE_PREMA,AUTRE_PRECIS,D_INVESTIGATEUR,NOM_INVESTIGATEUR,MOTIF_SORTIE_PREMA1
0,Subject Identifier for the Study,initiales patient,date de la visite de screening,année premiers symptomes,année du diagnostic de la maladie,année d'introduction de la L-DOPA,année d'introduction du traitement dopaminergique,année d'apparition des fluctuations motrices,année d'apparition des fluctuations non motrices,année d'apparition des dyskinésies,...,âge,sexe,suivi étude,date de fin de l'étude,date de sortie prématurée,motif de sortie prématurée,autre précision,date de signature de l'investigateur,nom de l'investigateur,motif de sortie prématurée LIB
1,01-001,SR,18/11/2013,1998,1999,2000,2000,2002,2000,.D,...,67,1,0,NaN,22/11/2013,1,NaN,18/03/2014,MOREAU CAROLINE,Patient non opéré
2,01-002,TM,13/01/2014,2006,2006,2006,2006,2011,.K,.K,...,59,1,0,NaN,20/02/2014,6,SUSPICION DE CANCER PULMONAIRE,24/03/2014,DR MOREAU,Autre
3,01-003,SJ,04/03/2014,2001,2001,2003,2001,2004,.K,2004,...,61,1,0,NaN,07/03/2014,1,NaN,21/03/2014,DR HOPES LUCIE,Patient non opéré
4,01-004,DJ,12/05/2014,1998,2000,2003,2000,2003,2003,2003,...,65,2,0,NaN,09/01/2015,6,RETRAIT DU MATERIEL ET RETRAIT DE CONSENTEMENT,09/01/2015,DEVOS,Autre


In [12]:
info.drop("D_SCREEN",axis=1,inplace=True)
info = nettoyer_codes_manquants(info)
info.to_excel("Output/version_3/info.xlsx",index=False)


  → codes remplacés par NaN : ['.K', '.D', '.A', '.C']


# Fusionner 

In [24]:
def reordonner_colonnes_par_variable(df):
    """
    Réorganise les colonnes du DataFrame fusionné de façon à grouper
    chaque variable ensemble sur toutes les visites.
    
    Ex: poids_V0, poids_V1, poids_V3 | taille_V0, taille_V1 | PDQ39_V0, PDQ39_V1 ...
    """
    import re

    visites = ["V0", "VC", "V1", "V3", "V5"]  # ordre des visites voulu

    # Séparer SUBJID du reste
    autres_cols = [c for c in df.columns if c != "SUBJID"]

    # Extraire le suffixe de visite d'une colonne
    def extraire_base_et_visite(col):
        for v in sorted(visites, key=len, reverse=True):  # tester les plus longs d'abord
            if col.endswith(f"_{v}"):
                base = col[:-(len(v) + 1)]  # retire "_Vx"
                return base, v
        return col, None  # colonne sans suffixe visite

    # Construire un dict : base → {visite: colonne}
    from collections import defaultdict, OrderedDict
    groupes = defaultdict(dict)
    sans_visite = []

    for col in autres_cols:
        base, visite = extraire_base_et_visite(col)
        if visite:
            groupes[base][visite] = col
        else:
            sans_visite.append(col)

    # Reconstruire l'ordre des colonnes :
    # Pour chaque base (dans l'ordre d'apparition original), 
    # mettre V0, VC, V1, V3, V5 côte à côte
    ordre_bases = list(dict.fromkeys(
        extraire_base_et_visite(c)[0] for c in autres_cols
        if extraire_base_et_visite(c)[1] is not None
    ))

    colonnes_ordonnees = ["SUBJID"]
    for base in ordre_bases:
        for v in visites:
            if v in groupes[base]:
                colonnes_ordonnees.append(groupes[base][v])

    # Ajouter les colonnes sans suffixe visite à la fin
    colonnes_ordonnees += sans_visite

    # Garder seulement les colonnes qui existent vraiment
    colonnes_ordonnees = [c for c in colonnes_ordonnees if c in df.columns]

    return df[colonnes_ordonnees]

In [30]:
def construire_df_final(dossier="."):
    """
    Lit directement les fichiers Excel de chaque visite,
    construit le tableau synthétique pour chaque visite,
    merge tout et réorganise les colonnes.
    """
    visites = [
        ("v0.xlsx", 0),
        ("vc.xlsx", "c"),
        ("v1.xlsx", 1),
        ("v3.xlsx", 3),
        ("v5.xlsx", 5),
    ]

    dfs = []
    for fichier, v in visites:
        chemin = os.path.join(dossier, fichier)
        if not os.path.exists(chemin):
            print(f"⚠️ Fichier manquant : {fichier}, ignoré.")
            continue
        
        # Lire toutes les feuilles d'un coup → dict {nom_feuille: DataFrame}
        result = pd.read_excel(chemin, sheet_name=None)
        
        df_v = construire_tableau_synthetique(result, v)
        dfs.append(df_v)

    # Merge tous les Vx sur SUBJID
    from functools import reduce
    df_all = reduce(lambda l, r: pd.merge(l, r, on="SUBJID", how="outer"), dfs)

    # Réorganiser les colonnes par variable
    df_final = reordonner_colonnes_par_variable(df_all)

    return df_final

In [31]:
import os
import pandas as pd

df_final = construire_df_final(dossier="Output/version_3")
df_final.to_excel("resultat_final.xlsx", index=False)

In [32]:
df_final.head()

,SUBJID,DATE_V0,DATE_V1,DATE_V3,DATE_V5,POIDS_V0,POIDS_V1,POIDS_V3,POIDS_V5,TAILLE_V0,...,DATE_Vc,Anxiolytiques_Vc,Antidépresseurs_Vc,Neuroleptiques_Vc,Thymorégulateurs_Vc,ledd_levot_Vc,ledd_agot_Vc,ledd_imaobt_Vc,ledd_mantt_Vc,ledd_tot_Vc
0,01-001,18/11/2013,NaN,NaN,NaN,76.0,NaN,NaN,NaN,171.0,...,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
1,01-002,13/01/2014,NaN,NaN,NaN,88.0,NaN,NaN,NaN,180.0,...,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
2,01-003,04/03/2014,NaN,NaN,NaN,89.0,NaN,NaN,NaN,176.0,...,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
3,01-004,12/05/2014,NaN,NaN,NaN,59.0,NaN,NaN,NaN,163.0,...,23/09/2014,2.0,1.0,0.0,0.0,700.00,120.0,NaN,NaN,820.00
4,01-005,16/06/2014,25/01/2016,17/04/2018,NaN,81.0,82.0,NaN,NaN,169.0,...,27/01/2015,1.0,0.0,0.0,0.0,893.75,NaN,NaN,NaN,893.75


In [33]:
df_info = pd.read_excel("Output/version_3/info.xlsx")

df_complet = pd.merge(df_info, df_final, on="SUBJID", how="outer")

# Mettre SUBJID en premier
cols = ["SUBJID"] + [c for c in df_complet.columns if c != "SUBJID"]
df_complet = df_complet[cols]

df_complet.to_excel("Output/version_3/data01.xlsx", index=False)